# **Revisi Generator Variasi Bahasa Lokal — Development v4**

---

**Proyek:** KP Agentic AI + LLM  
**Tahap:** Step 12 — Perbaikan variasi ID–EN dan ID–Jawa  
**Dataset:** MASSIVE-Agents `id-ID` development

Revisi v4 menggunakan instruksi induk yang sama dengan v2 dan v3. Fokusnya adalah memperluas variasi ID–EN dan ID–Jawa melalui penggantian kata atau frasa yang berkaitan dengan maksud instruksi. Data dan hasil versi sebelumnya tidak diubah.

## **1. Dataset dan Konfigurasi**

---

Dataset terdiri dari 397 instruksi Indonesia development. Kandidat induk dan hasil v3 digunakan untuk menjaga keterlacakan setiap perubahan. Seluruh proses menggunakan seed tetap.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

SEED = 20261110
TARGET_AUDIT = 100
ROOT_OVERRIDE = None
KATEGORI = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]
KATEGORI_LAMA = {"slang", "singkatan", "typo"}


def cari_root():
    candidates = []
    if ROOT_OVERRIDE is not None:
        candidates.append(Path(ROOT_OVERRIDE))
    if os.environ.get("KP_AGENTIC_AI_ROOT"):
        candidates.append(Path(os.environ["KP_AGENTIC_AI_ROOT"]))
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents, Path(r"D:\KP-Agentic-AI")])

    for root in candidates:
        data_path = root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
        v3_path = root / "results" / "variasi_bahasa_terkontrol" / "development_v3" / "kandidat_variasi_development_v3.csv"
        formal_path = root / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "kandidat_induk_formal_v2.csv"
        if data_path.is_file() and v3_path.is_file() and formal_path.is_file():
            return root
    raise FileNotFoundError("Dataset development, daftar induk v2, atau kandidat v3 belum ditemukan.")

ROOT = cari_root()
INPUT = ROOT / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
INPUT_V3 = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v3" / "kandidat_variasi_development_v3.csv"
INPUT_INDUK = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "kandidat_induk_formal_v2.csv"
OUTPUT = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v4"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Dataset:", INPUT.name)
print("Kandidat sebelumnya:", INPUT_V3.name)
print("Hasil v4:", OUTPUT)

Project: D:\KP-Agentic-AI
Dataset: id-ID_development.jsonl
Kandidat sebelumnya: kandidat_variasi_development_v3.csv
Hasil v4: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v4


## **2. Pemeriksaan Sumber Data**

---

Setiap instruksi induk dicocokkan dengan `row_key`, `group_id`, GT, dan split pada dataset sumber. Kandidat slang, singkatan, dan typo dari v3 tetap dibawa sebagai data yang belum divalidasi manusia.

In [2]:
def baca_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

with INPUT.open("r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

v3 = baca_csv(INPUT_V3)
induk_v2 = baca_csv(INPUT_INDUK)

assert len(data) == 397, "Jumlah instruksi development berubah."
assert len(v3) == 671, "Kandidat v3 berbeda dari hasil yang sudah diperiksa."
assert len(induk_v2) == 359, "Jumlah kandidat induk berbeda dari hasil sebelumnya."

sumber = {row["row_key"]: row for row in data}
assert len(sumber) == len(data)

induk = {}
for item in induk_v2:
    source = sumber[item["row_key_induk"]]
    assert item["instruksi_id_asli"] == source["question"]
    assert item["group_id"] == source["group_id"]
    assert source["split"] == "development"
    induk[source["row_key"]] = source

v3_by_id = {item["variant_id"]: item for item in v3}
assert len(v3_by_id) == len(v3)
for item in v3:
    source = induk[item["row_key_induk"]]
    assert item["group_id"] == source["group_id"]
    assert item["instruksi_id_asli"] == source["question"]
    assert json.loads(item["gt_json"]) == source["gt"]

print("Instruksi development:", len(data))
print("Kandidat induk:", len(induk))
print("Kandidat v3:", len(v3))

Instruksi development: 397
Kandidat induk: 359
Kandidat v3: 671


## **3. Aturan Code-Switching ID–EN**

---

Campuran Indonesia–Inggris dibentuk dengan mengganti tindakan seperti `putar` menjadi `play`, `cek` menjadi `check`, atau `kirim` menjadi `send`. Pola yang hanya menambahkan `please` tidak digunakan. Frasa yang berisiko kurang alami tetap diberi tanda untuk audit.

In [3]:
# Format: (pola, pengganti, subtipe, tingkat_review)
ATURAN_EN = [
    (r"\btolong\s+(?:putar|putarkan|mainkan)\b", "tolong play", "musik", "sedang"),
    (r"\btolong\s+(?:cek|periksa)\b", "tolong check", "periksa", "sedang"),
    (r"\btolong\s+(?:hapus|buang)\b", "tolong delete", "hapus", "sedang"),
    (r"\btolong\s+(?:atur|setel|pasang)\b", "tolong set", "jadwal", "sedang"),
    (r"\btolong\s+(?:kirim|kirimkan)\b", "tolong send", "pesan", "sedang"),
    (r"\btolong\s+(?:cari|carikan)\b", "tolong search", "pencarian", "sedang"),
    (r"\btolong\s+(?:tampilkan|tunjukkan|perlihatkan)\b", "tolong show", "tampilan", "sedang"),
    (r"\btolong\s+(?:nyalakan|hidupkan)\b", "tolong turn on", "perangkat", "sedang"),
    (r"\btolong\s+matikan\b", "tolong turn off", "perangkat", "sedang"),
    (r"\btolong\s+tambahkan\b", "tolong add", "tambah", "sedang"),
    (r"\btolong\s+buatkan\s+kopi\b", "tolong brew kopi", "kopi", "perlu_review_bahasa"),
    (r"\btolong\s+(?:buat|buatkan)\b", "tolong create", "buat", "perlu_review_bahasa"),
    (r"\btolong\s+ingatkan\b", "tolong remind", "ingatkan", "sedang"),
    (r"\btolong\s+bacakan\b", "tolong read", "baca", "sedang"),
    (r"\b(?:tunjukkan|tampilkan|perlihatkan)\s+saya\b", "show me", "tampilan", "sedang"),
    (r"\b(?:carikan)\s+saya\b", "tolong search", "pencarian", "perlu_review_konteks"),
    (r"(?<!daftar )\b(?:putar|putarkan|mainkan)\b", "play", "musik", "sedang"),
    (r"\b(?:cek|periksa)\b(?=\s+(?:email|surel|cuaca|alarm|semua|apakah)\b)", "check", "periksa", "sedang"),
    (r"\b(?:kirim|kirimkan)\b(?=\s+(?:email|surel|pesan)\b)", "send", "pesan", "sedang"),
    (r"\b(?:hapus|buang)\b(?=\s+(?:alarm|daftar|janji|kegiatan|acara|semua)\b)", "delete", "hapus", "sedang"),
    (r"\b(?:atur|setel|pasang)\b(?=\s+(?:alarm|pengingat|kalender|lampu|pertemuan)\b)", "set", "jadwal", "sedang"),
    (r"\btambahkan\b(?=\s+(?:rapat|makan|pengingat|surel|daftar|pembayaran)\b)", "add", "tambah", "sedang"),
    (r"\b(?:tampilkan|tunjukkan|perlihatkan)\b", "show", "tampilan", "perlu_review_bahasa"),
    (r"\bcari\s+tahu\s+tentang\b", "check info tentang", "pencarian", "perlu_review_bahasa"),
    (r"\b(?:cari|carikan)\b(?!\s+tahu\b)", "search", "pencarian", "perlu_review_bahasa"),
    (r"\bnyalakan\b(?=\s+(?:lampu|radio|steker|robot|teko|mesin)\b)", "turn on", "perangkat", "sedang"),
    (r"\bmatikan\b(?=\s+(?:lampu|speaker|alarm|volume|semua)\b)", "turn off", "perangkat", "sedang"),
    (r"\bmulai\b(?=\s+(?:radio|program|pandora|putar|lagu)\b)", "start", "mulai", "sedang"),
    (r"\bbuka\b(?=\s+(?:email|kontak|kotak|saga)\b)", "open", "buka", "sedang"),
    (r"\bingatkan\s+saya\b", "remind aku", "ingatkan", "perlu_review_ragam"),
    (r"\bingatkan\b(?=\s+(?:aku|untuk)\b)", "remind", "ingatkan", "sedang"),
    (r"\bbacakan\b(?=\s+(?:daftar|jadwal|saya|kiriman)\b)", "read", "baca", "sedang"),
    (r"\bbuat\b(?=\s+(?:email|daftar|acara)\b)", "create", "buat", "sedang"),
    (r"\blanjutkan\b(?=\s+(?:buku|laskar|bumi|episode)\b)", "continue", "lanjut", "sedang"),
]

print("Aturan ID–EN:", len(ATURAN_EN))

Aturan ID–EN: 34


## **4. Aturan Code-Switching ID–Jawa**

---

Campuran Indonesia–Jawa menggunakan sejumlah padanan ngoko untuk tindakan dan pertanyaan. Contohnya `matikan` menjadi `pateni` dan `bagaimana` menjadi `piye`. Seluruh hasil Jawa masih memerlukan pemeriksaan penutur karena pilihan kata dan ragam bahasa dapat berbeda antarwilayah.

In [4]:
ATURAN_JAWA = [
    (r"\btolong\s+(?:putar|putarkan|mainkan)\b", "tulung puterna", "musik", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:cek|periksa)\b", "tulung cekna", "periksa", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:hapus|buang)\b", "tulung busak", "hapus", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:atur|setel|pasang)\b", "tulung setelna", "jadwal", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:kirim|kirimkan)\b", "tulung kirimna", "pesan", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:cari|carikan)\b", "tulung golekna", "pencarian", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:tampilkan|tunjukkan|perlihatkan)\b", "tulung duduhna", "tampilan", "perlu_penutur_jawa"),
    (r"\btolong\s+(?:nyalakan|hidupkan)\b", "tulung uripna", "perangkat", "perlu_penutur_jawa"),
    (r"\btolong\s+matikan\b", "tulung pateni", "perangkat", "perlu_penutur_jawa"),
    (r"\btolong\s+tambahkan\b", "tulung tambahna", "tambah", "perlu_penutur_jawa"),
    (r"\btolong\s+ingatkan\b", "tulung elingna", "ingatkan", "perlu_penutur_jawa"),
    (r"\btolong\s+bacakan\b", "tulung wacakna", "baca", "perlu_penutur_jawa"),
    (r"\b(?:beri\s+tahu\s+saya|beritahu\s+saya)\b", "kandani aku", "informasi", "perlu_penutur_jawa"),
    (r"\b(?:tunjukkan|tampilkan|perlihatkan)\s+saya\b", "duduhna aku", "tampilan", "perlu_penutur_jawa"),
    (r"\b(?:apakah\s+ada|apa\s+ada|adakah)\b", "ono ora", "pertanyaan_ada", "perlu_penutur_jawa"),
    (r"\bbagaimana\s+cara\b", "piye carane", "pertanyaan_cara", "perlu_penutur_jawa"),
    (r"\bjam\s+berapa\b", "jam piro", "pertanyaan_waktu", "perlu_penutur_jawa"),
    (r"\bberapa\s+lama\b", "piro suwene", "pertanyaan_durasi", "perlu_penutur_jawa"),
    (r"\bberapa\s+harga\b", "piro regane", "pertanyaan_harga", "perlu_penutur_jawa"),
    (r"\bberapa\s+suhu\b", "suhune piro", "pertanyaan_suhu", "perlu_penutur_jawa"),
    (r"\bberapa\s+tahun\b", "piro taun", "pertanyaan_tahun", "perlu_penutur_jawa"),
    (r"\bada\s+berapa\b", "ono piro", "pertanyaan_jumlah", "perlu_penutur_jawa"),
    (r"\bapa\s+saja\b", "opo wae", "pertanyaan_daftar", "perlu_penutur_jawa"),
    (r"\bapa\s+yang\b", "opo sing", "pertanyaan_isi", "perlu_penutur_jawa"),
    (r"\bsiapa\s+yang\b", "sopo sing", "pertanyaan_orang", "perlu_penutur_jawa"),
    (r"\b(?:di\s+mana|dimana|dimanakah)\b", "neng endi", "pertanyaan_lokasi", "perlu_penutur_jawa"),
    (r"\b(?:matikan)\b", "pateni", "perangkat", "perlu_penutur_jawa"),
    (r"\b(?:nyalakan)\b", "uripna", "perangkat", "perlu_penutur_jawa"),
    (r"(?<!daftar )\b(?:putar|putarkan|mainkan)\b", "puterna", "musik", "perlu_penutur_jawa"),
    (r"\b(?:hapus|buang)\b", "busak", "hapus", "perlu_penutur_jawa"),
    (r"\b(?:atur|setel)\b", "setelna", "jadwal", "perlu_penutur_jawa"),
    (r"\b(?:kirim|kirimkan)\b", "kirimna", "pesan", "perlu_penutur_jawa"),
    (r"\b(?:tampilkan|tunjukkan|perlihatkan)\b", "duduhna", "tampilan", "perlu_penutur_jawa"),
    (r"\bcari\s+tahu\b", "golek info", "pencarian", "perlu_penutur_jawa"),
    (r"\bcarikan\s+saya\b", "tulung golekna", "pencarian", "perlu_penutur_jawa"),
    (r"\b(?:cari|carikan|temukan)\b", "golekna", "pencarian", "perlu_penutur_jawa"),
    (r"\bbacakan\b", "wacakna", "baca", "perlu_penutur_jawa"),
    (r"\bingatkan\s+saya\b", "elingna aku", "ingatkan", "perlu_penutur_jawa"),
    (r"\bingatkan\b", "elingna", "ingatkan", "perlu_penutur_jawa"),
    (r"\bbangunkan\b", "gugahna", "alarm", "perlu_penutur_jawa"),
    (r"\btambahkan\b", "tambahna", "tambah", "perlu_penutur_jawa"),
    (r"\bmulai\b", "wiwiti", "mulai", "perlu_penutur_jawa"),
    (r"\bbuka\b", "bukak", "buka", "perlu_penutur_jawa"),
    (r"\blanjutkan\b", "terusna", "lanjut", "perlu_penutur_jawa"),
    (r"\b(?:periksa|cek)\b", "cekna", "periksa", "perlu_penutur_jawa"),
    (r"\bbagaimana\b", "piye", "pertanyaan", "perlu_penutur_jawa"),
    (r"\bapa\b(?!\s+pun\b)", "opo", "pertanyaan", "perlu_penutur_jawa"),
    (r"\bberapa\b", "piro", "pertanyaan", "perlu_penutur_jawa"),
    (r"\bsiapa\b", "sopo", "pertanyaan", "perlu_penutur_jawa"),
]

print("Aturan ID–Jawa:", len(ATURAN_JAWA))

Aturan ID–Jawa: 49


## **5. Perlindungan Argumen GT**

---

Kata atau frasa yang muncul sebagai nilai argumen GT dilindungi ketika ditemukan di instruksi. Aturan tidak diterapkan jika menyentuh bagian tersebut. Pemeriksaan ini menjaga nilai yang terlihat secara teks, tetapi belum membuktikan bahwa makna seluruh variasi tetap sama.

In [5]:
def iter_nilai_teks(value):
    if isinstance(value, dict):
        for child in value.values():
            yield from iter_nilai_teks(child)
    elif isinstance(value, (list, tuple)):
        for child in value:
            yield from iter_nilai_teks(child)
    elif isinstance(value, str) and value.strip():
        yield value.strip()


def rentang_gt(row):
    text = row["question"]
    spans = []
    for value in set(iter_nilai_teks(row["gt"])):
        if value.lower() in {"current", "now", "null", "none"}:
            continue
        pattern = r"(?<!\w)" + re.escape(value) + r"(?!\w)"
        for match in re.finditer(pattern, text, re.I):
            spans.append((match.start(), match.end(), value))
    return spans


def ubah_terkontrol(row, aturan):
    original = row["question"]
    spans = rentang_gt(row)
    for rule_index, (pattern, replacement, subtype, flag) in enumerate(aturan, start=1):
        for match in re.finditer(pattern, original, re.I):
            if any(match.start() < right and match.end() > left for left, right, _ in spans):
                continue
            new_text = original[:match.start()] + replacement + original[match.end():]
            if new_text == original:
                continue
            if any(
                len(re.findall(re.escape(value), new_text, re.I)) <
                len(re.findall(re.escape(value), original, re.I))
                for _, _, value in spans
            ):
                continue
            return new_text, match.group(), replacement, subtype, flag, rule_index
    return None

print("Perlindungan nilai GT: aktif")

Perlindungan nilai GT: aktif


## **6. Pembentukan Kandidat v4**

---

Kategori slang, singkatan, dan typo tetap memakai hasil v3. Kategori ID–EN dan ID–Jawa dibentuk kembali menggunakan aturan baru. Setiap induk hanya menghasilkan satu kandidat pada setiap kategori, dengan GT dan split yang tetap.

In [6]:
hasil = []
perubahan = []
tidak_terbentuk = []

for row_key, parent in induk.items():
    for cat in KATEGORI:
        variant_id = f"{row_key}::{cat}"
        old = v3_by_id.get(variant_id)
        if cat in KATEGORI_LAMA:
            if old is not None:
                item = dict(old)
                item["versi_generator"] = "v3_dipertahankan_belum_validasi"
                item["status_validasi"] = "BELUM_VALIDASI_MANUSIA"
                item["flag_review"] = "perlu_validasi_manusia"
                hasil.append(item)
            continue

        aturan = ATURAN_EN if cat == "code_switch_id_en" else ATURAN_JAWA
        candidate = ubah_terkontrol(parent, aturan)
        if candidate is None:
            tidak_terbentuk.append({
                "row_key_induk": row_key,
                "kategori": cat,
                "alasan": "tidak ada pola yang cocok di luar nilai GT",
            })
            if old is not None:
                perubahan.append({
                    "variant_id": variant_id, "kategori": cat,
                    "teks_v3": old["instruksi_variasi"], "teks_v4": "",
                    "status": "TIDAK_TERBENTUK", "alasan": "aturan v4 tidak cocok",
                })
            continue

        text_new, original_span, replacement, subtype, flag, rule_index = candidate
        item = {
            "variant_id": variant_id,
            "row_key_induk": parent["row_key"],
            "source_record_id": parent["source_record_id"],
            "group_id": parent["group_id"],
            "split": parent["split"],
            "language_code": parent["language_code"],
            "kategori": cat,
            "subtipe": subtype,
            "tingkat_review": flag,
            "instruksi_id_asli": parent["question"],
            "instruksi_variasi": text_new,
            "aturan_asal": original_span,
            "aturan_pengganti": replacement,
            "fungsi_gt": next(iter(parent["gt"])),
            "gt_json": json.dumps(parent["gt"], ensure_ascii=False, sort_keys=True),
            "status_validasi": "BELUM_VALIDASI_MANUSIA",
            "versi_generator": "v4_rule_based",
            "flag_review": flag,
            "rule_index": rule_index,
        }
        hasil.append(item)
        perubahan.append({
            "variant_id": variant_id, "kategori": cat,
            "teks_v3": old["instruksi_variasi"] if old else "",
            "teks_v4": text_new,
            "status": "BARU" if old is None else ("BERUBAH" if old["instruksi_variasi"] != text_new else "SAMA"),
            "alasan": "pola tindakan/pertanyaan terkontrol",
        })

assert len({row["variant_id"] for row in hasil}) == len(hasil)
print("Kandidat v3:", len(v3))
print("Kandidat v4:", len(hasil))
for cat in KATEGORI:
    print(f"{cat:<22} {sum(row['kategori'] == cat for row in hasil)}")

Kandidat v3: 671
Kandidat v4: 971
slang                  139
singkatan              187
typo                   304
code_switch_id_en      128
code_switch_id_jawa    213


## **7. Pemeriksaan Konsistensi**

---

Pemeriksaan memastikan `row_key`, fungsi GT, isi GT, instruksi asli, `group_id`, dan split tetap sama. Kandidat yang tidak berbeda dari instruksi induk tidak diterima. Pemeriksaan ini tidak menggantikan validasi makna dan penutur.

In [7]:
checks = []
for item in hasil:
    original = sumber[item["row_key_induk"]]
    has_original_gt_text = all(
        (not re.search(r"(?<!\w)" + re.escape(value) + r"(?!\w)", original["question"], re.I)) or
        bool(re.search(r"(?<!\w)" + re.escape(value) + r"(?!\w)", item["instruksi_variasi"], re.I))
        for value in iter_nilai_teks(original["gt"])
        if value.lower() not in {"current", "now", "null", "none"}
    )
    checks.append({
        "variant_id": item["variant_id"],
        "gt_sama": json.loads(item["gt_json"]) == original["gt"],
        "fungsi_sama": item["fungsi_gt"] == next(iter(original["gt"])),
        "group_id_sama": item["group_id"] == original["group_id"],
        "split_development": item["split"] == original["split"] == "development",
        "instruksi_induk_sama": item["instruksi_id_asli"] == original["question"],
        "variasi_berbeda": item["instruksi_variasi"] != original["question"],
        "nilai_gt_terlihat_tetap": has_original_gt_text,
    })

assert all(all(value for key, value in row.items() if key != "variant_id") for row in checks)
assert len({row["group_id"] for row in induk.values()}) == len(induk)
print("Pemeriksaan struktural:", "PASS")
print("Validasi semantik:", "BELUM")

Pemeriksaan struktural: PASS
Validasi semantik: BELUM


## **8. Sampel Audit dan Ringkasan Kategori**

---

Sampel diambil maksimal 100 per kategori. Pengambilan dibuat tersebar menurut subtipe aturan, lalu dibatasi satu `group_id` untuk tiap kategori. Kekurangan sampel dicatat tanpa membuat variasi tambahan yang dipaksakan.

In [8]:
rng = random.Random(SEED)
sampel = []
ringkasan = []

for cat in KATEGORI:
    items = [dict(row) for row in hasil if row["kategori"] == cat]
    buckets = defaultdict(list)
    for item in items:
        buckets[item["subtipe"]].append(item)
    for bucket in buckets.values():
        rng.shuffle(bucket)

    selected = []
    used_groups = set()
    # Putaran per subtipe mengurangi dominasi satu jenis aturan.
    while len(selected) < TARGET_AUDIT and any(buckets.values()):
        for subtype in sorted(buckets):
            bucket = buckets[subtype]
            while bucket:
                row = bucket.pop()
                if row["group_id"] not in used_groups:
                    used_groups.add(row["group_id"])
                    selected.append(row)
                    break
            if len(selected) >= TARGET_AUDIT:
                break

    for row in selected:
        sampel.append({
            **row,
            "makna_tetap": "BELUM_DIREVIEW",
            "entitas_waktu_tetap": "BELUM_DIREVIEW",
            "bahasa_alami": "BELUM_DIREVIEW",
            "fungsi_gt_masih_sesuai": "BELUM_DIREVIEW",
            "keputusan_manusia": "BELUM_DIREVIEW",
            "catatan_manusia": "",
        })

    jumlah_v3 = sum(row["kategori"] == cat for row in v3)
    ringkasan.append({
        "kategori": cat,
        "kandidat_v3": jumlah_v3,
        "kandidat_v4": len(items),
        "perubahan": len(items) - jumlah_v3,
        "sampel_audit": len(selected),
        "target_audit": TARGET_AUDIT,
        "kekurangan": max(0, TARGET_AUDIT - len(selected)),
        "subtipe_terwakili": len({item["subtipe"] for item in selected}),
    })

for item in ringkasan:
    print(f'{item["kategori"]:<22} v3={item["kandidat_v3"]:<4} v4={item["kandidat_v4"]:<4} audit={item["sampel_audit"]:<4} kurang={item["kekurangan"]}')
print("Total sampel audit:", len(sampel))

slang                  v3=139  v4=139  audit=100  kurang=0
singkatan              v3=187  v4=187  audit=100  kurang=0
typo                   v3=304  v4=304  audit=100  kurang=0
code_switch_id_en      v3=20   v4=128  audit=100  kurang=0
code_switch_id_jawa    v3=21   v4=213  audit=100  kurang=0
Total sampel audit: 500


## **9. Penyimpanan Hasil**

---

Seluruh hasil v4 disimpan dalam folder `development_v4`. Dataset sumber, hasil v2/v3, dan prediksi model tidak ditimpa. Kolom keputusan validasi manusia tetap kosong sampai pemeriksaan independen dilakukan.

In [9]:
def simpan_csv(rows, filename):
    path = OUTPUT / filename
    if not rows:
        return None
    keys = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)
    return path

kamus = []
for cat, rules in [("code_switch_id_en", ATURAN_EN), ("code_switch_id_jawa", ATURAN_JAWA)]:
    for number, (pattern, replacement, subtype, flag) in enumerate(rules, start=1):
        kamus.append({
            "kategori": cat, "rule_index": number,
            "pola": pattern, "pengganti": replacement,
            "subtipe": subtype, "tingkat_review": flag,
        })

files = [
    simpan_csv(hasil, "kandidat_variasi_development_v4.csv"),
    simpan_csv(perubahan, "perubahan_dari_v3.csv"),
    simpan_csv(tidak_terbentuk, "kategori_tidak_terbentuk_v4.csv"),
    simpan_csv(checks, "pemeriksaan_konsistensi_v4.csv"),
    simpan_csv(sampel, "sampel_validasi_v4.csv"),
    simpan_csv(ringkasan, "ringkasan_v3_vs_v4.csv"),
    simpan_csv(kamus, "kamus_aturan_v4.csv"),
]
metadata = {
    "run": "revisi_variasi_bahasa_development_v4",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source_sha256": hashlib.sha256(INPUT.read_bytes()).hexdigest(),
    "input_v3_sha256": hashlib.sha256(INPUT_V3.read_bytes()).hexdigest(),
    "input_development_rows": len(data),
    "parent_rows": len(induk),
    "variants_v3": len(v3),
    "variants_v4": len(hasil),
    "audit_rows": len(sampel),
    "counts": {r["kategori"]: r["kandidat_v4"] for r in ringkasan},
    "audit_shortfall": {r["kategori"]: r["kekurangan"] for r in ringkasan},
    "gt_modified": False,
    "test_used": False,
    "generator_model_used": False,
    "structural_checks": "PASS",
    "semantic_validation": "pending_human_review",
    "native_javanese_review": "pending",
    "approved_for_final_model_evaluation": False,
    "note": "Variasi berbasis aturan belum otomatis setara secara semantik atau alami.",
}
metadata_path = OUTPUT / "metadata_development_v4.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
files.append(metadata_path)

for file in files:
    if file:
        print(file.name)
print("Total file:", len([file for file in files if file]))

kandidat_variasi_development_v4.csv
perubahan_dari_v3.csv
kategori_tidak_terbentuk_v4.csv
pemeriksaan_konsistensi_v4.csv
sampel_validasi_v4.csv
ringkasan_v3_vs_v4.csv
kamus_aturan_v4.csv
metadata_development_v4.json
Total file: 8


## **10. Kesimpulan Sementara**

---

Perubahan dari v3 ke v4 ditujukan untuk memperbanyak variasi code-switching yang memiliki perubahan tindakan atau pertanyaan, bukan sekadar penambahan penanda kesopanan. GT dan split tetap dipertahankan. Hasil ini masih berupa kandidat yang memerlukan validasi manual sebelum digunakan sebagai dataset evaluasi akhir.